In [1]:
from pyspark.sql import SparkSession
from minio import Minio

spark = SparkSession.builder.appName("nyc-taxi-parquet").getOrCreate()

# Spark uses Hadoop configs internally to connect to storage systems like S3 / MinIO
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin123")
hadoop_conf.set("fs.s3a.path.style.access", "true")        # Required for MinIO because it uses path-style URLs instead of virtual-hosted style
                                                           # Example path-style: http://minio:9000/bucket-name/object
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")  # MinIO is running on HTTP locally (not HTTPS)

In [2]:
# ---------- Connect to MinIO ----------

minio_client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

bucket_name = "nyc-taxi"

# Print all objects present inside the bucket
# This helps verify whether parquet files exist and are accessible
print("\n" + "="*60)
print(f"Objects inside '{bucket_name}':")
for obj in minio_client.list_objects(bucket_name, recursive=True):
    print(f" - {obj.object_name} ({obj.size} bytes)")
print("="*60)
print("NYC Taxi Parquet Upload Complete")
print("="*60)


Objects inside 'nyc-taxi':
 - 2023/yellow_tripdata_2023-01.parquet (47673370 bytes)
 - 2023/yellow_tripdata_2023-02.parquet (47748012 bytes)
 - 2023/yellow_tripdata_2023-03.parquet (56127762 bytes)
 - 2023/yellow_tripdata_2023-04.parquet (54222699 bytes)
 - 2023/yellow_tripdata_2023-05.parquet (58654627 bytes)
 - 2023/yellow_tripdata_2023-06.parquet (54999465 bytes)
 - 2023/yellow_tripdata_2023-07.parquet (48361828 bytes)
 - 2023/yellow_tripdata_2023-08.parquet (48152353 bytes)
 - 2023/yellow_tripdata_2023-09.parquet (47895515 bytes)
 - 2023/yellow_tripdata_2023-10.parquet (59009059 bytes)
 - 2023/yellow_tripdata_2023-11.parquet (56094653 bytes)
 - 2023/yellow_tripdata_2023-12.parquet (56804275 bytes)
 - 2024/yellow_tripdata_2024-01.parquet (49961641 bytes)
 - 2024/yellow_tripdata_2024-02.parquet (50349284 bytes)
 - 2024/yellow_tripdata_2024-03.parquet (60078280 bytes)
 - 2024/yellow_tripdata_2024-04.parquet (59133625 bytes)
 - 2024/yellow_tripdata_2024-05.parquet (62553128 bytes)
 - 

In [3]:
from pyspark.sql.functions import col

# Month 1 (January) has a different schema than the other files. Therefore, some of its columns' data types have been changed.
# Also, the Airport_fee column name was different, so it has been changed as well.

# Read January separately and cast to match the standard schema
df_jan = spark.read.parquet("s3a://nyc-taxi/2023/yellow_tripdata_2023-01.parquet")

df_jan = df_jan \
    .withColumn("VendorID", col("VendorID").cast("integer")) \
    .withColumn("passenger_count", col("passenger_count").cast("long")) \
    .withColumn("RatecodeID", col("RatecodeID").cast("long")) \
    .withColumn("PULocationID", col("PULocationID").cast("integer")) \
    .withColumn("DOLocationID", col("DOLocationID").cast("integer")) \
    .withColumnRenamed("airport_fee", "Airport_fee")

# Read the rest (Feb-Dec)
df_rest = spark.read.parquet(
    "s3a://nyc-taxi/2023/yellow_tripdata_2023-0[2-9].parquet",
    "s3a://nyc-taxi/2023/yellow_tripdata_2023-1[0-2].parquet"
)

# Union them together
df1 = df_jan.unionByName(df_rest)

In [4]:
# Combining parquet files from all the folders to create a single dataframe.

from pyspark.sql.functions import lit

df1 = spark.read.parquet("s3a://nyc-taxi/2023/*.parquet")
df2 = spark.read.parquet("s3a://nyc-taxi/2024/*.parquet")
df_1_2 = df1.union(df2)

# Newer congestion charge (column name - cbd_congestion_fee) was introduced in 2025's datasets. 
# So, to union it with previous years' dataframes, we created the same column in previous years' dataframes.

df_1_2 = df_1_2.withColumn('cbd_congestion_fee', lit(0.0))
df3 = spark.read.parquet("s3a://nyc-taxi/2025/*.parquet")
df = df_1_2.union(df3)
df.count()

111036384